# AI in Medical Physics — Week 0: Colab + GitHub onboarding

**Estimated time:** 45–75 minutes  
**Compute:** Free Google Colab CPU runtime; no GPU required  
**Data:** Synthetic X-ray-like phantom images, generated in this notebook

### Learning outcomes
1. Run a reproducible Python notebook in a cloud environment.
2. Generate and inspect a simple imaging dataset.
3. Fit and evaluate a baseline classifier on a held-out test set.
4. Observe how a change in acquisition conditions can affect model performance.
5. Save your finished notebook to a **private** student GitHub repository.

> **Educational simulation only.** These images are not real radiographs, and the model is not clinically validated. Never upload patient images, PHI, passwords, or access tokens to this notebook.

## Part 1 — Verify your environment

Use **Runtime → Change runtime type → CPU**, then **Runtime → Run all**. The first cell reports available Python packages. Google Colab normally preinstalls the required libraries.

In [ ]:
import sys
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("scikit-learn:", sklearn.__version__)
try:
    import torch
    print("PyTorch:", torch.__version__, "| GPU available:", torch.cuda.is_available())
except ImportError:
    print("PyTorch optional for this exercise; not installed")

## Part 2 — Generate an X-ray-like phantom

We approximate the transmitted photon count at each pixel using a simple attenuation law:

$$\mathbb{E}[N(x,y)] = N_0\, \exp\bigl[-\mu T(x,y)-\Delta\mu L(x,y)\bigr].$$

Here $T$ describes background phantom thickness and $L$ is a small target region present only in positive examples. Observed counts are Poisson sampled. We then apply an approximate negative log transform.

**Limitations:** This toy forward model omits scatter, focal-spot/detector blur, correlated noise, exposure control, display processing, anatomical variation, and many other important effects.

In [ ]:
SEED = 2026
rng = np.random.default_rng(SEED)

# Fixed geometry for this simple onboarding exercise.
yy, xx = np.mgrid[-1:1:64j, -1:1:64j]
radius = np.sqrt(xx**2 + yy**2)
thickness = 1.6 * np.sqrt(np.clip(1 - (radius / 0.85)**2, 0, None)) * (radius < 0.85)
target_mask = ((xx - 0.16)**2 + (yy + 0.10)**2) < 0.15**2

def simulate_image(target_present, incident_photons, generator):
    if incident_photons <= 0:
        raise ValueError("incident_photons must be positive")
    delta_mu = 0.085 if target_present else 0.0
    expected_counts = incident_photons * np.exp(-0.45 * thickness - delta_mu * target_mask)
    counts = generator.poisson(expected_counts)
    return -np.log((counts + 0.5) / (incident_photons + 0.5))

def make_dataset(n, incident_photons, generator):
    if n % 2 != 0:
        raise ValueError("Use an even sample size for balanced classes")
    labels = np.tile([0, 1], n // 2)
    generator.shuffle(labels)
    images = np.stack([simulate_image(bool(label), incident_photons, generator)
                       for label in labels])
    return images, labels

images, labels = make_dataset(n=400, incident_photons=250, generator=rng)
print("Image shape:", images.shape, "Label shape:", labels.shape)
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
for ax, label in zip(axes, (0, 1)):
    example = images[np.flatnonzero(labels == label)[0]]
    ax.imshow(example, cmap="gray", vmin=0, vmax=0.85)
    ax.set_title("Target absent" if label == 0 else "Target present")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Part 3 — Fit a baseline classifier

We block-average each 64 × 64 image to 16 × 16 features, then train regularized logistic regression. Training and held-out test images have **independent noise realizations**; the test labels are not used for training.

In actual clinical studies, **patient-level, institution-level, and temporal separation** may also be essential. Image-level random splitting alone is often insufficient.

In [ ]:
def extract_features(images):
    # Non-overlapping 4x4 average blocks: 64x64 -> 16x16
    return images.reshape(-1, 16, 4, 16, 4).mean(axis=(2, 4)).reshape(-1, 256)

train_idx, test_idx = train_test_split(
    np.arange(len(labels)), test_size=0.30, stratify=labels, random_state=42
)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1200, C=0.5))
model.fit(extract_features(images[train_idx]), labels[train_idx])

reference_prob = model.predict_proba(extract_features(images[test_idx]))[:, 1]
reference_auc = roc_auc_score(labels[test_idx], reference_prob)
print(f"Reference-exposure held-out AUROC: {reference_auc:.3f}")

## Part 4 — Stress-test a changed acquisition condition

We evaluate the *same frozen model* on newly generated images with fewer incident photons. This is a simple example of a distribution shift: the object is unchanged, but measurement statistics change.

**Exercise:** Run once at `shift_photons = 25`, then change it to `75`, then `250`. Record the AUROC results and consider the variability from sampling and finite test size.

In [ ]:
shift_photons = 25  # TODO: Try 75 and 250, re-running this cell each time.
shift_images, shift_labels = make_dataset(
    n=120, incident_photons=shift_photons, generator=np.random.default_rng(9001)
)
shift_prob = model.predict_proba(extract_features(shift_images))[:, 1]
shift_auc = roc_auc_score(shift_labels, shift_prob)
print(f"Reference exposure: AUROC = {reference_auc:.3f}")
print(f"Changed exposure ({shift_photons} photons): AUROC = {shift_auc:.3f}")

base_fpr, base_tpr, _ = roc_curve(labels[test_idx], reference_prob)
shift_fpr, shift_tpr, _ = roc_curve(shift_labels, shift_prob)
plt.figure(figsize=(5, 5))
plt.plot(base_fpr, base_tpr, label=f"Reference (AUC={reference_auc:.2f})")
plt.plot(shift_fpr, shift_tpr, label=f"Changed (AUC={shift_auc:.2f})")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Model performance under acquisition shift")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

## Part 5 — Questions for discussion

Edit the text below. Write 1–3 sentences per question.

**Q1.** Report the changed-exposure AUROCs for 25, 75, and 250 incident photons. What happens, and why?

**Q2.** Name at least **two** limitations of the simplified phantom or detector model.

**Q3.** Give an example of *data leakage* caused by an inappropriate train/test split in a real medical imaging dataset.

**Q4.** Before deploying a clinical tool with similar intended use, name **two** local acceptance or commissioning tests you would require.

**Q5.** If the tool passes acceptance testing, what changes in clinical operations should trigger revalidation?

### Student answers

**Q1:** _Replace with your answer._

**Q2:** _Replace with your answer._

**Q3:** _Replace with your answer._

**Q4:** _Replace with your answer._

**Q5:** _Replace with your answer._

## Part 6 — Save and submit

1. **Do not edit the public course template.** Use a separate **private student repository**.
2. In Colab, choose **File → Save a copy in GitHub**; select your private repository and save to `notebooks/week00_completed.ipynb`.
3. If the private repository is unavailable in Colab, use **File → Download → Download .ipynb**, then upload the notebook to your private repository using GitHub's web interface.
4. Reopen the notebook from your repository to confirm that the executed results and written answers are present.
5. Submit the private notebook link in Canvas, after granting your instructor access.

**No patient data, institutional credentials, API keys, or protected files should be uploaded to personal Colab or GitHub accounts.**